# 18b · The Phase 4 threshold rule applied to the original recipe

After the 13-fold evidence (`17b`, section 4; `17c`), the project owner decided to use the **original recipe** (notebook 08) for the final system, not variant A. This notebook applies **the pre-declared Phase 4 choice rule, unchanged**, to the original recipe's 13 leave-one-prompt-out score sets (`results/16b_lopo`, seed 42), and uses whatever it picks. Nothing about the rules, targets, statistics, bootstrap or choice rule from `18_robust_threshold.ipynb` sections 1-6 is changed; the choice is not made between (b) and (d) on variant A's numbers. The held-out prompts and held-out generators are not scored here.

For reference, the declared rule (`18_robust_threshold.ipynb`, section 5): decide at the **1% target**; qualifying rules are those whose point estimate of mean AI caught is within **1.0 point of rule (a)**; choose the qualifying rule with the **smallest share of topics over target**, ties to the higher mean AI caught; if no rule other than (a) qualifies, or (a) is the choice, report the trade-off and ask.

New in this notebook, as the owner requested: **P(AI | essay in the cannot-tell band)**, at the dataset's AI share and at lower AI shares (below), and band wording if most essays in the band are human.

Part of the declaration for the P(AI | zone) calculation (written before computing it): for a zone with share $h$ of human essays and share $a$ of AI essays (leave-one-topic-out, equal weight per topic, as in the band table of notebook 18),
$$P(\mathrm{AI}\mid\text{zone}) = \frac{\pi\,a}{\pi\,a + (1-\pi)\,h},$$
with the AI prior $\pi$ = the dataset's AI share (34.6%, as in train, val and test), and also 10% and 5%. This is Bayes' rule and assumes the essays arriving later have the same human and AI shares in each zone as these 13 topics; the prior is the only thing that changes. Intervals: the same bootstrap as before.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.data import load_splits
from src.conformal import load_topics, compare_rules, band_report, set_tau, _p_arrays, loto, RULES

SEED, N_BOOT = 42, 1000
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()
topics_O = load_topics('../results/16b_lopo', range(13), data)
print('original recipe, 13 left-out topics:', [t['name'][:16] for t in topics_O])

## 1 · The four rules, leave-one-topic-out, original recipe: the full table (1% and 0.5% targets)

In [ ]:
cmp_O = compare_rules(topics_O, targets=(0.01, 0.005), n_boot=N_BOOT, seed=SEED)
cmp_O.round(4).to_csv('../results/18b_rule_comparison_original.csv', index=False)
def fmt(df):
    out = pd.DataFrame({'rule': df['rule'], 'target': df['target'].map('{:.1%}'.format)})
    for k, name in [('mean_fpr', 'mean false flags %'), ('worst_fpr', 'worst topic false flags %'), ('share_over_target', 'topics over target %'), ('mean_ai_caught', 'mean AI caught %')]:
        out[name] = [f"{r[k]:.2f} [{r[k + '_lo']:.2f}, {r[k + '_hi']:.2f}]" for _, r in df.iterrows()]
    return out.set_index(['target', 'rule'])
fmt(cmp_O)

## 2 · The pre-declared choice, applied unchanged

In [ ]:
c1 = cmp_O[cmp_O['target'] == 0.01].set_index('rule')
ref = c1.loc['a', 'mean_ai_caught']
qual = [r for r in RULES if c1.loc[r, 'mean_ai_caught'] >= ref - 1.0]
print(f"mean AI caught under rule (a): {ref:.2f}; qualifying rules (within 1.0 point): {qual}")
for r in RULES:
    print(f"   ({r}) mean AI caught {c1.loc[r, 'mean_ai_caught']:.3f}  ({c1.loc[r, 'mean_ai_caught'] - ref:+.3f} vs (a)), topics over target {c1.loc[r, 'share_over_target']:.1f}%")
best = sorted(qual, key=lambda r: (c1.loc[r, 'share_over_target'], -c1.loc[r, 'mean_ai_caught']))[0]
print(f"smallest share of topics over target among the qualifying rules: rule ({best}): {c1.loc[best, 'share_over_target']:.1f}% of topics over target")
if set(qual) == {'a'} or best == 'a':
    print('=> ONLY rule (a) qualifies or (a) is the choice: the declared procedure says report the trade-off and ASK.')
else:
    print(f'=> CHOSEN RULE for the original recipe: ({best})')
CHOSEN = best

### The 13 topics one by one, at the 1% target

In [ ]:
rows = []
pO = _p_arrays(topics_O)
for r in RULES:
    per = loto(pO, r, 0.01)
    for t, x in zip(topics_O, per.itertuples()):
        rows.append({'rule': r, 'topic': t['name'], 'false flags %': 100 * x.fpr, 'AI caught %': 100 * x.ai_caught})
bt = pd.DataFrame(rows)
bt.round(3).to_csv('../results/18b_by_topic_1pct_original.csv', index=False)
tab = bt.pivot(index='topic', columns='rule', values='false flags %'); tab['humans'] = [len(t['left_h']) for t in topics_O]
tab.round(2)

## 3 · The cannot-tell band with the chosen rule (edges as declared: flag at the rule's 1% threshold, cannot tell up to its 5% threshold)

In [ ]:
band = band_report(topics_O, CHOSEN, 0.01, 0.05, n_boot=N_BOOT, seed=SEED)
band.round(2).to_csv('../results/18b_band_original.csv')
print(f'rule ({CHOSEN}), original recipe, leave-one-topic-out over 13 topics')
band.round(2)

In [ ]:
sens = {f'{a:.1%} to {b:.0%}': band_report(topics_O, CHOSEN, a, b, n_boot=200, seed=SEED)['estimate'] for a, b in [(0.005, 0.05), (0.01, 0.10)]}
pd.DataFrame(sens).round(2)

## 4 · P(AI | zone): how much of each zone is AI, at different AI shares

Zones: **flagged** ($p \le \tau_{1\%}$), **cannot tell** ($\tau_{1\%} < p \le \tau_{5\%}$), **not flagged** ($p > \tau_{5\%}$), from the chosen rule, leave-one-topic-out. Priors: the dataset's AI share (34.6%), 10% and 5%.

In [ ]:
PRIORS = {'34.6% (the dataset)': 0.346, '10%': 0.10, '5%': 0.05}
def zone_shares(per):
    h = {'flagged': per['fpr'].mean(), 'cannot tell': per['human_in_band'].mean(), 'not flagged': per['human_not_flagged'].mean()}
    a = {'flagged': per['ai_caught'].mean(), 'cannot tell': per['ai_in_band'].mean(), 'not flagged': per['ai_missed'].mean()}
    return h, a
def p_ai(h, a, prior):
    return {z: prior * a[z] / (prior * a[z] + (1 - prior) * h[z]) if (prior * a[z] + (1 - prior) * h[z]) > 0 else np.nan for z in h}
rng = np.random.default_rng(SEED)
draws = {(z, k): [] for z in ('flagged', 'cannot tell', 'not flagged') for k in PRIORS}
for _ in range(N_BOOT):
    per = loto(_p_arrays(topics_O, rng), CHOSEN, 0.01, 0.05)
    h, a = zone_shares(per)
    for k, pi in PRIORS.items():
        for z, v in p_ai(h, a, pi).items(): draws[(z, k)].append(v)
h0, a0 = zone_shares(loto(pO, CHOSEN, 0.01, 0.05))
rows = []
for k, pi in PRIORS.items():
    pt = p_ai(h0, a0, pi)
    for z in ('flagged', 'cannot tell', 'not flagged'):
        d = np.array(draws[(z, k)], dtype=float); d = d[~np.isnan(d)]
        rows.append({'AI share of arriving essays': k, 'zone': z, 'P(AI | zone) %': f'{100 * pt[z]:.1f} [{100 * np.percentile(d, 2.5):.1f}, {100 * np.percentile(d, 97.5):.1f}]'})
pai = pd.DataFrame(rows).pivot(index='zone', columns='AI share of arriving essays', values='P(AI | zone) %').loc[['flagged', 'cannot tell', 'not flagged']][list(PRIORS)]
pai.to_csv('../results/18b_p_ai_by_zone_original.csv')
print('share of humans per zone:', {z: f'{100 * v:.2f}%' for z, v in h0.items()}, '| share of AI per zone:', {z: f'{100 * v:.2f}%' for z, v in a0.items()})
pai

## 5 · The thresholds for Phase 5, on the p scale, from all 13 topics (no topic left out)

In [ ]:
all_ph = [p[0] for p in _p_arrays(topics_O)]
final = pd.DataFrame({f'{t:.1%}': {r: set_tau(r, all_ph, t) for r in RULES} for t in (0.005, 0.01, 0.05)})
final.index.name = 'rule'
final.round(5).to_csv('../results/18b_final_thresholds_p_scale_original.csv')
print(f'tau on the p scale from all 13 topics, original recipe. Chosen rule: ({CHOSEN}).')
final.round(5)